In [ ]:
import time
import threading
import queue
import json

import smbus2
import Adafruit_DHT
import RPi.GPIO as GPIO
import yagmail
import paho.mqtt.client as mqtt

#SENSOR POSITIONS ON BBOARD
DHT_SENSOR = Adafruit_DHT.DHT11
DHT_PIN = 17

FLAME_PIN = 27
BUZZER_PIN = 22

I2C_BUS = 1
PCF8591_ADDR = 0x48

MQTT_BROKER = "localhost"
MQTT_PORT = 1883
MQTT_TOPIC = "hazard/data"

#ADD WHOEVER ELSE U WANNA SHOW THIS TOOOOO 

SENDER_EMAIL = "valeriestoes69@gmail.com"
SENDER_APP_PASSWORD = "spqk gkqv ukyq jfdc"
RECIPIENT_EMAILS = [
    "farahsalah101@gmail.com",
    "ehunter25@gwmail.gwu.edu"
]

EMAIL_COOLDOWN = 300
last_email_time = 0
last_status_sent = None


GPIO.setmode(GPIO.BCM)
GPIO.setup(FLAME_PIN, GPIO.IN)
GPIO.setup(BUZZER_PIN, GPIO.OUT)
GPIO.output(BUZZER_PIN, GPIO.LOW)

bus = smbus2.SMBus(I2C_BUS)

mqtt_client = mqtt.Client()
mqtt_client.connect(MQTT_BROKER, MQTT_PORT, 60)

latest_temp = 25.0
latest_hum = 40.0

email_queue = queue.Queue()








def read_adc(channel):
    bus.write_byte(PCF8591_ADDR, 0x40 | channel)
    bus.read_byte(PCF8591_ADDR)
    return bus.read_byte(PCF8591_ADDR)

def clamp(x):
    return max(0.0, min(1.0, x))

def norm_gas(x):
    return clamp(x / 255.0)

def norm_temp(x):
    return clamp((x - 30.0) / 20.0)

def norm_hum(x):
    if x < 20:
        return clamp((20 - x) / 20.0)
    if x > 75:
        return clamp((x - 75) / 25.0)
    return 0.0

def classify(score, flame, temp, hum):
    if flame:
        return "CRITICAL"
#ALSO 90 DEG F
    if temp >= 32.0:
        return "CRITICAL"
#80% HUMIDITY IS BAD, CHANGE DEPENDING ON HOW IT FUNCTIONS
    if hum >= 80.0:
        return "CRITICAL"
#if the score is over 0.45, then at least 2 sensors are critical. 
    if score >= 0.40:
        return "WARNING"
    return "SAFE"

def humiture_worker():
    global latest_temp, latest_hum
    while True:
        hum, temp = Adafruit_DHT.read(DHT_SENSOR, DHT_PIN)
        if temp is not None:
            latest_temp = temp
        if hum is not None:
            latest_hum = hum
        time.sleep(3)

def email_worker():
    while True:
        payload = email_queue.get()
        if payload is None:
            break
        try:
            yag = yagmail.SMTP(SENDER_EMAIL, SENDER_APP_PASSWORD)
            yag.send(
                to=RECIPIENT_EMAILS,
                subject=f"Hazard Alert: {payload['status']}",
                contents=f"""
Hazard detected

Time: {payload['timestamp']}
Status: {payload['status']}
Gas: {payload['gas']}
Temperature: {payload['temp']} C
Temperature: {payload['temp_F']} F
Humidity: {payload['hum']} %
Flame detected: {payload['flame']}
Score: {payload['score']}
"""
            )
            print("EMAIL SENT")
        except Exception as e:
            print("Email failed:", e)

def maybe_send_email(payload):
    global last_email_time, last_status_sent

    now = time.time()

    if payload["status"] not in ("WARNING", "CRITICAL"):
        if payload["status"] == "SAFE":
            last_status_sent = None
        return

    if now - last_email_time < EMAIL_COOLDOWN:
        return

    if last_status_sent == payload["status"]:
        return

    email_queue.put(payload)
    last_email_time = now
    last_status_sent = payload["status"]


threading.Thread(target=humiture_worker, daemon=True).start()
threading.Thread(target=email_worker, daemon=True).start()


print("System running...")

try:
    while True:
        gas = read_adc(0)
        flame = GPIO.input(FLAME_PIN) == 1

        temp = latest_temp
        hum = latest_hum
#I changed this based on my importance scale A FIRE IS MORE IMP THAN HUMIDITY
        score = (
            0.2 * norm_gas(gas) +
            0.3 * norm_temp(temp) +
            0.2 * norm_hum(hum) +
            0.3 * (1 if flame else 0)
        )

        status = classify(score, flame, temp, hum)

        GPIO.output(BUZZER_PIN, status != "SAFE")

        payload = {
            "timestamp": time.strftime("%H:%M:%S"),
            "gas": round(gas, 2),
            "temp": round(temp, 2),
            "temp_F": round((temp * 9 / 5) + 32, 2),
            "hum": round(hum, 2),
            "flame": flame,
            "score": round(score, 3),
            "status": status
        }

        print(payload)

        mqtt_client.publish(MQTT_TOPIC, json.dumps(payload))
        maybe_send_email(payload)

        time.sleep(0.5)

except KeyboardInterrupt:
    print("Stopping...")

finally:
    GPIO.output(BUZZER_PIN, GPIO.LOW)
    GPIO.cleanup()
    bus.close()
    email_queue.put(None)
    mqtt_client.disconnect()